# Sports Vision: Detecting Judo Throws with RNNs & LSTMs 🥋

- **Contact:** hello@edinburghai.org
- **Credits:** Created by Conor O'Shea for Edinburgh AI. Please credit us if you reuse this workshop!

---

# What's this notebook about? 🤔

Imagine a coach has several full judo matches to review. Before giving feedback, they first need to find the moments worth watching.

**What is judo? 🥋**

Judo is a combat sport where two athletes try to throw each other onto their backs. They can also win through groundwork, such as pins or submissions, but today we'll focus on throws.

A **throw attempt** is when an athlete tries to execute a throw, whether or not they successfully bring their opponent down. These are useful moments for a coach to review.

**How can we turn a video into a sequence? 🎞️**

A video is simply a sequence of still images, called **frames**, shown one after another. At 30 frames per second, a seven-second clip contains **210 frames**.

Rather than treating those frames as separate photographs, we can treat them as an **ordered sequence** that shows how the athletes move over time. The order matters: the frames show the setup, movement and outcome of an action.

Today we'll build models that take this sequence of 210 frames, represented using the athletes' pose coordinates, and predict whether the clip contains **at least one throw attempt**.

We'll compare a basic **Recurrent Neural Network (RNN)** with a **Long Short-Term Memory network (LSTM)** using pose data from the Judo Clipper research project.

**No judo knowledge needed!** The main idea is that we can view video as a sequence and use sequence models to learn from movement over time. This also applies to other sports and human activities.

## The structure of this notebook 📖

Work through Tasks 0–7 in order. Each task explains the idea before asking you to use it. In the student version, complete the sections marked `# TODO` and `...`.

You should be comfortable with Python and have seen a basic neural-network training loop. We'll use NumPy, Pandas, PyTorch, scikit-learn and Matplotlib.

The optional extension introduces bidirectional LSTMs once the main practical is complete.

# Task 0: Set up the notebook and add the data 📂

We'll run this workshop on Kaggle so everyone has the same environment.

**Your Job**

1. Go to **Kaggle.com** and sign in.
2. Create a new notebook.
3. Download the student notebook shared in the workshop WhatsApp group.
4. Use **File → Import Notebook** and open the downloaded `.ipynb` file.
5. In the sidebar, open **Input → Add Input** and add the [**Version 2 RNN/LSTM Workshop Judo Data**](https://www.kaggle.com/datasets/conoroshea46/version2-rnn-lstm-workshop-judo-data).
6. In notebook settings, select a **GPU accelerator** if one is available.

Make sure you attach this **version 2 dataset**. The old Judo Throw Interval Dataset contains different labels and won't work here.

The next cell finds the new dataset by its files, so it doesn't depend on the eventual Kaggle dataset slug. For local use, set `DATASET_DIR` to the extracted dataset folder.

In [ ]:
from pathlib import Path
import copy
import random
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from IPython.display import Image, Video, display

DATASET_DIR = None  # Local example: Path("judo-clip-workshop-data")
if DATASET_DIR is None:
    candidates = [
        path.parent for path in Path("/kaggle/input").rglob("lstm_inputs.npy")
        if (path.parent / "split_manifest.csv").is_file()
    ]
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Attach the new workshop dataset, or set DATASET_DIR explicitly. "
            f"Found {len(candidates)} matching folders."
        )
    DATASET_DIR = candidates[0]
DATASET_DIR = Path(DATASET_DIR)

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(min(2, torch.get_num_threads()))
print("Dataset:", DATASET_DIR)
print("Device:", device)

# Section 1: Turning sport into sequences 📈

## What does our model actually see?

We can watch a match as video, but today's recurrent models receive **joint coordinates**, rather than pixels.

A pose estimator finds 17 keypoints for each athlete, including shoulders, elbows, hips and ankles. Each keypoint has an `x` and a `y` coordinate:

- One athlete: `17 × 2 = 34` numbers per frame.
- Two athletes: `34 × 2 = 68` numbers per frame.
- Seven seconds at 30 frames per second: `7 × 30 = 210` frames.

One clip therefore becomes an array with shape **`[210, 68]`**. Coordinates are normalised relative to the image dimensions. They still contain information about camera framing and where athletes appear in the image.

## Why does order matter?

A single bent-over pose might occur while gripping, entering a throw or recovering from one. The movements before and after it help distinguish these situations. Our model can learn from that ordered sequence.

The next GIF comes from the **original workshop** and illustrates poses moving over time. Its annotations refer to the earlier per-frame detection task and its poses came from the older extraction pipeline. We'll preserve it as an illustration, while training on the newer data below.

In [ ]:
display(Image(filename=str(DATASET_DIR / "attempt_id490_annotated.gif"), width=600))

## Task 1: Watch examples and inspect the arrays 👀

First compare a throw-attempt clip with a no-attempt clip. Both are training examples, so we can inspect them before model selection.

Our inputs have shape `[number_of_clips, time_steps, features]`. Our labels have shape `[number_of_clips]`, because we're predicting **one thing about the whole clip**.

**Your Job**

Load the two NumPy arrays, print their shapes and check how many examples belong to each class. `.npy` files store NumPy arrays directly, so `np.load()` gives us the numerical data.

In [ ]:
for clip_id in ["attempt_id960", "no_throw_clip_2451"]:
    print(clip_id)
    display(Video(str(DATASET_DIR / "examples" / f"{clip_id}.mp4"), embed=True, width=600))

In [ ]:
inputs = np.load(DATASET_DIR / "lstm_inputs.npy", allow_pickle=False)
labels = np.load(DATASET_DIR / ..., allow_pickle=False)  # TODO: Load the label file.
manifest = pd.read_csv(DATASET_DIR / "training_manifest.csv")

print("Inputs:", ...)  # TODO: Display the input array shape.
print("Labels:", ...)  # TODO: Display the label array shape.
print("Class counts:", dict(zip(*np.unique(labels, return_counts=True))))

assert inputs.shape == (2163, 210, 68)
assert labels.shape == (2163,)
assert np.isfinite(inputs).all()
assert np.isin(labels, [0, 1]).all()
assert np.array_equal(manifest["array_row_idx"], np.arange(len(inputs)))
assert manifest["clip_id"].is_unique

## Where did these poses come from?

The research pipeline uses YOLO11x-pose and ByteTrack, followed by a Viterbi-based player-selection system to choose the two competitors across the clip. Short gaps in keypoints are interpolated, poor sequences are rejected, and remaining unresolved coordinates become zero at export.

We've already done this expensive processing. You won't need to run YOLO or implement tracking today. The arrays contain **1,390 no-attempt clips and 773 attempt clips**.

A zero-filled missing coordinate still limits what the classifier can learn. Preprocessing makes the data usable, but doesn't make every pose reliable.

# Task 2: Prepare training, validation and test data 🔀

- **Training:** update the model's learned parameters.
- **Validation:** select checkpoints, compare models and choose thresholds.
- **Test:** evaluate after those decisions are complete.

## Why can't we tune using the test data? ⚠️

The test set is meant to give us a fair final check on data that hasn't influenced our choices. If we inspect its results, change the model to improve them and test again, we're indirectly fitting our decisions to that test set. The reported result can then look better than performance on genuinely new data.

**Use the validation set to choose hyperparameters, compare RNNs against LSTMs, select a checkpoint and choose a classification threshold.** This includes settings such as hidden size, number of layers, dropout, learning rate and number of epochs.

**Keep the test set untouched until you've finished making those choices.** A fixed random seed doesn't make test-based tuning fair.


We'll reuse the research project's saved split so everyone works with the same examples: **1,730 training, 216 validation and 217 test clips**.

The split file identifies each clip's array row and label. Keeping those rows aligned matters: otherwise we'd train the model against the wrong answers.

**One limitation:** the split separates clips, rather than entire source matches. Clips from the same bout can appear across splits. Results therefore measure performance on unseen clips, and don't establish performance on entirely unseen matches.

**Your Job**

Select the row indices for each split. Use the provided checks to confirm that the saved split matches our arrays.

In [ ]:
splits = pd.read_csv(DATASET_DIR / "split_manifest.csv").sort_values("array_row_idx")
assert np.array_equal(splits["array_row_idx"], np.arange(len(inputs)))
assert np.array_equal(splits["clip_id"], manifest["clip_id"])
assert np.array_equal(splits["label"], labels)
assert splits["split"].isin(["train", "validation", "test"]).all()

train_indices = splits.loc[
    splits["split"] == ...,  # TODO: Select training rows.
    "array_row_idx",
].to_numpy()
val_indices = splits.loc[
    splits["split"] == ...,  # TODO: Select validation rows.
    "array_row_idx",
].to_numpy()
test_indices = splits.loc[
    splits["split"] == ...,  # TODO: Select test rows.
    "array_row_idx",
].to_numpy()

print("Train / validation / test:", len(train_indices), len(val_indices), len(test_indices))

## Batching our sequences 🚚

A PyTorch `Dataset` pairs each input with its label. A `DataLoader` groups those pairs into batches.

All our sequences already have 210 frames, so we can use `TensorDataset` without writing a custom padding function. Labels become floating-point tensors for the binary loss function.

We shuffle **which clips** appear together during training. We keep the **frames inside each clip** in chronological order.

In [ ]:
BATCH_SIZE = 32

def make_loader(indices, shuffle=False):
    dataset = TensorDataset(
        torch.from_numpy(inputs[indices]),
        torch.from_numpy(labels[indices]).float(),
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(train_indices, shuffle=...)  # TODO: Should training clip order be shuffled?
val_loader = make_loader(val_indices)
test_loader = make_loader(test_indices)

batch_inputs, batch_labels = next(iter(train_loader))
print("Batch inputs:", batch_inputs.shape)  # [32, 210, 68]
print("Batch labels:", batch_labels.shape)  # [32]

# Section 2: Learning from movement 🧠

## Task 3: Build a basic RNN

An RNN processes one frame vector at a time. It maintains a **hidden state**, a learned numerical summary of the sequence so far.

At each time step it combines the new frame with the previous hidden state:

$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$$

Here, `x_t` contains the current frame's 68 features. The hidden state's size is a model setting, rather than something fixed by the number of joints.

After processing all 210 frames, we take the **last layer's final hidden state** and pass it through a small feed-forward classification head. That produces one raw score, called a **logit**, for the clip.

**Your Job**

Configure the RNN's input size and select the final hidden state in `forward()`. `batch_first=True` tells PyTorch that inputs use `[batch, time, features]` ordering.

In [ ]:
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.30

class ThrowRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=...,  # TODO: How many features describe one frame?
            hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS, dropout=DROPOUT, batch_first=True,
        )
        self.head = nn.Sequential(
            nn.Linear(HIDDEN_SIZE, 64), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(64, 1),
        )

    def forward(self, sequence):
        _, hidden_state = self.rnn(sequence)
        clip_representation = ...  # TODO: Select the final hidden state of the last layer.
        return self.head(clip_representation).squeeze(-1)

seed_everything()
rnn_model = ThrowRNN().to(device)
with torch.no_grad():
    print("RNN output shape:", rnn_model(batch_inputs.to(device)).shape)

## Task 4: Build an LSTM

During training, gradients must travel backwards through the recurrent computations. Over many time steps, they can become very small or very large, which can make learning long-range relationships difficult.

An LSTM adds a **cell state** and learned gates that control which information to retain, add and expose:

- **Forget gate:** scales information already in the cell state.
- **Input gate:** scales new candidate information added to it.
- **Output gate:** controls what the cell state exposes through the hidden state.

The cell-state update contains an additive path:

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

That path can help preserve information and gradients, especially when the forget gate stays near one. LSTMs still require suitable training settings and don't guarantee better results on every dataset.

PyTorch implements the gate calculations inside `nn.LSTM`. Unlike `nn.RNN`, it returns both hidden and cell states: `output, (h_n, c_n)`.

**Your Job**

Configure the LSTM and use its final hidden state to classify the clip. We'll keep the same classification head and training settings as our RNN, so the recurrence is the main architectural difference. The models still have different parameter counts.

In [ ]:
class ThrowLSTM(nn.Module):
    def __init__(self, bidirectional=False):
        super().__init__()
        self.bidirectional = bidirectional
        self.lstm = nn.LSTM(
            input_size=...,  # TODO: Use the same frame features as the RNN.
            hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS, dropout=DROPOUT,
            batch_first=True, bidirectional=bidirectional,
        )
        directions = 2 if bidirectional else 1
        self.head = nn.Sequential(
            nn.Linear(HIDDEN_SIZE * directions, 64), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(64, 1),
        )

    def forward(self, sequence):
        _, (hidden_state, cell_state) = self.lstm(sequence)
        if self.bidirectional:
            # Final forward and backward states from the last LSTM layer.
            clip_representation = torch.cat((hidden_state[-2], hidden_state[-1]), dim=1)
        else:
            clip_representation = ...  # TODO: Select the final hidden state of the last layer.
        return self.head(clip_representation).squeeze(-1)

seed_everything()
lstm_model = ThrowLSTM().to(device)
with torch.no_grad():
    print("LSTM output shape:", lstm_model(batch_inputs.to(device)).shape)
for name, model in [("RNN", rnn_model), ("LSTM", lstm_model)]:
    print(name, "parameters:", sum(p.numel() for p in model.parameters()))

# Task 5: Train the models 🏋️

## How do we measure an incorrect prediction?

We'll use **binary cross-entropy with logits**, because each clip has a yes/no label. Give `BCEWithLogitsLoss` raw logits: it handles the sigmoid calculation internally.

There are more negative clips than positive clips. We'll weight positive examples by `number_of_training_negatives / number_of_training_positives`, using **training data only**. This makes positive errors contribute more to the training objective.

For checkpoint selection, we'll calculate ordinary **unweighted validation loss**, as in the research pipeline. Training and validation losses therefore use different weighting and shouldn't be compared as if their scales were identical.

**Your Job**

Define the positive-class weight and binary loss. The training helper is provided so we can concentrate on sequence modelling.

After `loss.backward()`, we clip the gradient norm to 1.0 before the optimiser step. This limits unusually large updates. It doesn't fix vanishing gradients or establish that a loss spike was caused by exploding gradients.

## Why train for 40 epochs?

We'll give each model **40 epochs**, but keep a copy of the checkpoint with the **lowest validation loss**. The best model might occur well before epoch 40, so we restore that checkpoint after training rather than automatically using the final epoch.

The training output reports the time per epoch. Use that to estimate the remaining time on your machine.


In [ ]:
train_labels = labels[train_indices]
positive_weight = ...  # TODO: Divide the training negative count by the training positive count.
train_criterion = nn.BCEWithLogitsLoss(  # TODO: Why do we pass logits rather than sigmoid outputs?
    pos_weight=torch.tensor([positive_weight], device=device),
)
val_criterion = nn.BCEWithLogitsLoss()
print("Positive-class weight:", round(positive_weight, 4))

# Train for 40 epochs and restore the checkpoint with the lowest validation loss.
# After epoch 1, use its measured duration to estimate the remaining time.
EPOCHS = 40
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4

In [ ]:
def train_model(model, name, epochs=EPOCHS):
    seed_everything()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    best_loss = float("inf")
    best_state = None
    history = []
    for epoch in range(1, epochs + 1):
        started = time.perf_counter()
        model.train()
        train_total = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = train_criterion(logits, y)
            if not torch.isfinite(loss):
                raise ValueError("Non-finite training loss")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_total += loss.item() * len(x)

        model.eval()
        val_total = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                val_total += val_criterion(model(x), y).item() * len(x)
        train_loss = train_total / len(train_loader.dataset)
        val_loss = val_total / len(val_loader.dataset)
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        seconds = time.perf_counter() - started
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "seconds": seconds})
        print(f"{name} {epoch}/{epochs}: train={train_loss:.4f}, val={val_loss:.4f}, {seconds:.1f}s")

    # Restore a copy of the best checkpoint, rather than the final epoch.
    model.load_state_dict(best_state)
    return pd.DataFrame(history)

In [ ]:
rnn_history = train_model(rnn_model, "RNN")
lstm_history = train_model(lstm_model, "LSTM")


# Task 6: Compare the models using validation data 📊

The model outputs a logit. Applying sigmoid turns it into a score between zero and one. A threshold converts that score into a binary decision.

We'll choose each model's threshold using **validation attempt F1**, and compare the models using validation results. **Do all hyperparameter tuning and model comparison here using validation data. Do not run the test evaluation to decide which settings to try next.** Keep the test set for the final step.

- **Precision:** among flagged clips, what fraction really contain attempts?
- **Recall:** among attempt clips, what fraction does the model flag?
- **F1:** combines precision and recall into one score.

For a coach, false positives mean extra footage to review. False negatives mean missed attempts. Changing the threshold lets us explore that tradeoff.

**Your Job**

Convert logits into scores and scores into predictions. Then use the plots and results to discuss which model works better in this run.

In [ ]:
def predict_scores(model, loader):
    model.eval()
    scores = []
    with torch.no_grad():
        for x, _ in loader:
            logits = model(x.to(device))
            scores.append(
                ...  # TODO: Apply sigmoid, move to CPU and convert to a NumPy array.
            )
    return np.concatenate(scores)

def classification_metrics(y_true, scores, threshold):
    predictions = ...  # TODO: Predict 1 for scores at or above the threshold.
    return {
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "attempt_f1": f1_score(y_true, predictions, zero_division=0),
        "macro_f1": f1_score(y_true, predictions, average="macro", zero_division=0),
    }

def select_threshold(y_true, scores):
    rows = [dict(threshold=float(t), **classification_metrics(y_true, scores, t))
            for t in np.linspace(0, 1, 101)]
    # Same tie-breaking priorities as the research implementation.
    table = pd.DataFrame(rows)
    best = table.sort_values(
        ["attempt_f1", "recall", "precision", "macro_f1", "threshold"],
        ascending=[False, False, False, False, True],
    ).iloc[0]
    return float(best["threshold"]), table

models = {"RNN": rnn_model, "LSTM": lstm_model}
thresholds, threshold_tables, validation_scores = {}, {}, {}
validation_rows = []
for name, model in models.items():
    scores = predict_scores(model, val_loader)
    validation_scores[name] = scores
    threshold, table = select_threshold(labels[val_indices], scores)
    thresholds[name], threshold_tables[name] = threshold, table
    validation_rows.append(dict(model=name, threshold=threshold,
        **classification_metrics(labels[val_indices], scores, threshold)))
validation_results = pd.DataFrame(validation_rows).set_index("model")
display(validation_results.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name, history in zip(axes, ["RNN", "LSTM"], [rnn_history, lstm_history]):
    ax.plot(history["epoch"], history["train_loss"], label="Training (weighted)")
    ax.plot(history["epoch"], history["val_loss"], label="Validation (unweighted)")
    ax.set(title=name, xlabel="Epoch", ylabel="Loss")
    ax.legend()
plt.tight_layout()
plt.show()

for name, table in threshold_tables.items():
    plt.plot(table["threshold"], table["attempt_f1"], label=name)
plt.xlabel("Threshold")
plt.ylabel("Validation attempt F1")
plt.legend()
plt.show()

# A simple reference: always predict the most frequent training class.
majority_class = int(np.bincount(train_labels).argmax())
majority_predictions = np.full(len(val_indices), majority_class)
print("Majority baseline validation accuracy:", accuracy_score(labels[val_indices], majority_predictions))
print("Majority baseline attempt F1:", f1_score(labels[val_indices], majority_predictions, zero_division=0))

## What can we conclude? 🤔

Look at the results you actually obtained:

- Does either model beat the majority baseline on attempt F1?
- Does one model recover more attempts at the cost of more false positives?
- Does validation loss stop improving while training loss continues falling?
- Would a coach prefer a higher-recall threshold, even if precision falls?

Loss curves alone don't prove the model learned judo mechanics, and an unstable curve doesn't diagnose a specific gradient problem. A single seed also doesn't establish that one architecture is generally superior.

You can change settings and repeat training while comparing **validation** results. Once you're happy with the configuration, continue to the final test. These workshop models don't reproduce the final bidirectional research model's reported F1.

# Task 7: Final test and video predictions 🎬

We'll select the model with higher validation attempt F1 and keep its validation-selected threshold. Then we'll evaluate it on the test set.

**Your Job**

Run the final evaluation and interpret the confusion matrix. **After seeing the test results, don't change hyperparameters, switch models, choose another epoch or adjust the threshold to improve this test score.** That would make the evaluation unfair because the test set would have influenced your decisions. Further development should use validation results, with a fresh untouched test set for a new final evaluation.

In [ ]:
selected_name = validation_results["attempt_f1"].idxmax()
selected_model = models[selected_name]
selected_threshold = thresholds[selected_name]
test_scores = predict_scores(selected_model, test_loader)
print("Selected model:", selected_name, "threshold:", selected_threshold)
display(pd.Series(classification_metrics(labels[test_indices], test_scores, selected_threshold)).round(3))
ConfusionMatrixDisplay.from_predictions(
    labels[test_indices], (test_scores >= selected_threshold).astype(int),
    labels=[0, 1], display_labels=["No attempt", "Attempt"], colorbar=False,
)
plt.title("Final test results")
plt.show()

# This example belongs to the test split. Its score comes from the test pass above.
clip_id = "attempt_id967"
array_row = int(manifest.loc[manifest["clip_id"] == clip_id, "array_row_idx"].iloc[0])
test_position = int(np.flatnonzero(test_indices == array_row)[0])
score = float(test_scores[test_position])
print(f"{clip_id}: label={labels[array_row]}, score={score:.3f}, prediction={int(score >= selected_threshold)}")
display(Video(str(DATASET_DIR / "examples" / f"{clip_id}.mp4"), embed=True, width=600))

# Save the teaching model for later use with the same input format.
checkpoint_path = Path("judo_workshop_model.pt")
torch.save({
    "model_type": selected_name, "state_dict": selected_model.state_dict(),
    "threshold": selected_threshold, "hidden_size": HIDDEN_SIZE,
    "num_layers": NUM_LAYERS, "dropout": DROPOUT, "bidirectional": False,
    "input_shape": [210, 68],
}, checkpoint_path)
print("Saved:", checkpoint_path.resolve())

# Optional extension: A bidirectional LSTM 🔁

We classify a completed clip, so every frame is available before we make a decision. A bidirectional LSTM processes it in both directions and combines their final hidden states.

Our `ThrowLSTM` class already supports this. In a bidirectional model, `hidden_state[-2]` and `hidden_state[-1]` are the final forward and backward states of the last layer. Concatenating them gives a vector with `2 × HIDDEN_SIZE` features.

**Your Job**

Train this variant and compare its **validation** result with the unidirectional LSTM. Ideally, do this before Task 7 if you intend to include it in final model selection. If you've already inspected test results, treat this as further validation exploration and avoid making a fresh generalisation claim from repeated use of that test set.

In [ ]:
RUN_BIDIRECTIONAL_EXTENSION = False
if RUN_BIDIRECTIONAL_EXTENSION:
    seed_everything()
    bidirectional_model = ThrowLSTM(bidirectional=True).to(device)
    bidirectional_history = train_model(bidirectional_model, "Bidirectional LSTM")
    scores = predict_scores(bidirectional_model, val_loader)
    threshold, _ = select_threshold(labels[val_indices], scores)
    display(pd.Series(classification_metrics(labels[val_indices], scores, threshold)).round(3))

# What could we investigate next?

An entire match could be analysed using overlapping windows like these, then the positive intervals could become review clips. Today's notebook covers the sequence classifier within that larger pipeline.

Useful next experiments include testing on entirely unseen matches, evaluating university club footage, or comparing pose extraction quality. Our inputs come from broadcast competition footage, which may differ from the footage a club coach records.

The model can also be sensitive to player assignment errors, camera cuts and missing poses. A binary label that ignores which athlete attempts the throw **doesn't make the neural network automatically invariant to swapping the two players**.

## Credits and licence

Workshop and dataset preparation: Conor O'Shea, Edinburgh AI. Pose extraction uses Ultralytics YOLO11x-pose and ByteTrack. The practical uses PyTorch, NumPy, Pandas, scikit-learn and Matplotlib.

Copyright 2026 Conor Ryan O'Shea. Workshop code is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0). Video footage retains its original third-party rights.